# 🐟 Notebook 01 — Exploratory Data Analysis
## Issaquah Creek Salmon Return Study | Summer 2025
**Friends of the Issaquah Salmon Hatchery (FISH)**

This notebook covers:
1. Load and validate the master dataset
2. 40-year time-series plots of Chinook and Coho returns
3. Stressor variable trend visualization
4. Correlation matrices
5. Mann-Kendall trend tests
6. Key EDA findings summary

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import pymannkendall as mk
from pathlib import Path

from src.data_pipeline import (
    build_master_dataset, validate_master,
    BASELINE_2025, START_YEAR, END_YEAR
)
from src.features import build_features

# Plot style
plt.rcParams.update({
    'font.family':    'DejaVu Sans',
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'figure.dpi':     120,
})

FIG_DIR = Path('../outputs/figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)

BLUE_DARK  = '#1F4E79'
BLUE_MID   = '#2E75B6'
SALMON     = '#C55A11'
GREEN      = '#375623'

print('Setup complete.')

## 1. Load the Master Dataset

In [ ]:
# ── Option A: Build fresh from raw files ──────────────────────────────────
# df_raw = build_master_dataset(
#     escapement_file  ='../data/raw/wdfw_issaquah_escapement.csv',
#     hatchery_file    ='../data/raw/fish_hatchery_releases.csv',
#     impervious_file  ='../data/raw/king_county_impervious.csv',
#     fetch_live_data  = True,
# )

# ── Option B: Load saved master dataset ───────────────────────────────────
df_raw = pd.read_csv('../data/processed/issaquah_creek_master.csv')

# Run all feature engineering
df = build_features(df_raw)

print(f'\nDataset shape: {df.shape}')
print(f'Years covered: {df.water_year.min()} – {df.water_year.max()}')
df.head()

In [ ]:
# Validate and check 2025 anchor point
checks = validate_master(df)

print('\n--- 2025 Baseline (from FISH annual report) ---')
for k, v in BASELINE_2025.items():
    print(f'  {k}: {v:,}')

## 2. 40-Year Return Trend — Both Species

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 9), sharex=True)

for ax, (species, col, color) in zip(axes, [
    ('Chinook (King)', 'chinook_total', BLUE_DARK),
    ('Coho (Silver)',  'coho_total',    SALMON),
]):
    roll_col = col.replace('_total', '_roll5')

    ax.bar(df['water_year'], df[col], color=color, alpha=0.35,
           label='Annual Return', width=0.8)
    if roll_col in df.columns:
        ax.plot(df['water_year'], df[roll_col], color=color, linewidth=2.5,
                label='5-Year Rolling Avg')

    # Highlight 2025 anchor
    val_2025 = df.loc[df['water_year'] == 2025, col]
    if len(val_2025) > 0:
        ax.scatter([2025], val_2025.values, color=color, s=100,
                   zorder=5, label=f'2025: {val_2025.values[0]:,.0f}')

    ax.set_ylabel('Returns (fish)', fontsize=11)
    ax.set_title(f'Issaquah Creek — {species} Annual Returns ({START_YEAR}–{END_YEAR})',
                 fontsize=12, fontweight='bold')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    ax.legend(fontsize=10)

axes[-1].set_xlabel('Water Year', fontsize=11)

plt.suptitle('Issaquah Creek Salmon Returns | Friends of the Issaquah Salmon Hatchery',
             y=1.01, fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / '01_return_trends.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → 01_return_trends.png')

## 3. Stressor Variables — Trend Overlay

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()

stressors = [
    ('swe_apr1_in',        'April 1 Snowpack (inches SWE)',       '#4472C4'),
    ('mean_water_temp_c',  'Mean Annual Water Temp (°C)',          '#C00000'),
    ('impervious_pct',     'Impervious Surface Coverage (%)',      '#7030A0'),
    ('pdo_winter_mean',    'PDO Winter Index (ocean conditions)',  '#00B0F0'),
]

for ax, (col, label, color) in zip(axes, stressors):
    if col not in df.columns:
        ax.text(0.5, 0.5, f'{col}\n(data not yet loaded)',
                ha='center', va='center', transform=ax.transAxes, color='#999')
        ax.set_title(label)
        continue

    ax.plot(df['water_year'], df[col], color=color, linewidth=2)
    ax.fill_between(df['water_year'], df[col], alpha=0.15, color=color)

    # Add trend line
    valid = df[['water_year', col]].dropna()
    if len(valid) > 5:
        z = np.polyfit(valid['water_year'], valid[col], 1)
        p = np.poly1d(z)
        ax.plot(valid['water_year'], p(valid['water_year']),
                'k--', linewidth=1, alpha=0.5, label=f'Trend: {z[0]:+.3f}/yr')
        ax.legend(fontsize=9)

    ax.set_title(label, fontsize=11, fontweight='bold')
    ax.set_xlabel('Water Year', fontsize=10)

plt.suptitle('Issaquah Creek Watershed — Key Stressor Trends',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(FIG_DIR / '02_stressor_trends.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → 02_stressor_trends.png')

## 4. Correlation Heatmap — Returns vs. Stressors

In [ ]:
corr_cols = [
    'chinook_total', 'coho_total',
    'swe_apr1_in', 'swe_anomaly_pct',
    'mean_flow_cfs', 'min_summer_flow_cfs',
    'mean_water_temp_c', 'max_summer_temp_c', 'days_above_18c',
    'pdo_winter_mean', 'pdo_lag1_winter',
    'impervious_pct', 'impervious_5yr_growth',
    'climate_stress_index',
]
corr_cols = [c for c in corr_cols if c in df.columns]

corr = df[corr_cols].corr(method='spearman')   # Spearman: robust to outliers

# Clean display names
name_map = {
    'chinook_total':         'Chinook Returns',
    'coho_total':            'Coho Returns',
    'swe_apr1_in':           'Snowpack (Apr 1)',
    'swe_anomaly_pct':       'Snowpack Anomaly %',
    'mean_flow_cfs':         'Mean Streamflow',
    'min_summer_flow_cfs':   'Min Summer Flow',
    'mean_water_temp_c':     'Mean Water Temp',
    'max_summer_temp_c':     'Max Summer Temp',
    'days_above_18c':        'Days >18°C',
    'pdo_winter_mean':       'PDO Winter',
    'pdo_lag1_winter':       'PDO (1-yr lag)',
    'impervious_pct':        'Impervious %',
    'impervious_5yr_growth': 'Imperv. 5yr Growth',
    'climate_stress_index':  'Climate Stress Index',
}
corr.columns = [name_map.get(c, c) for c in corr.columns]
corr.index   = [name_map.get(c, c) for c in corr.index]

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    corr, annot=True, fmt='.2f', cmap='RdBu_r',
    vmin=-1, vmax=1, center=0,
    square=True, linewidths=0.5,
    cbar_kws={'shrink': 0.8},
    ax=ax
)
ax.set_title('Spearman Correlation Matrix — Issaquah Creek Salmon & Stressors',
             fontsize=12, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig(FIG_DIR / '03_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → 03_correlation_heatmap.png')

# Print top correlations with Chinook
print('\nTop correlations with Chinook returns (Spearman):')
print(corr['Chinook Returns'].drop('Chinook Returns').sort_values(key=abs, ascending=False).to_string())

## 5. Mann-Kendall Trend Tests

In [ ]:
mk_targets = {
    'Chinook Returns':        'chinook_total',
    'Coho Returns':           'coho_total',
    'April 1 Snowpack (SWE)': 'swe_apr1_in',
    'Mean Water Temp (°C)':   'mean_water_temp_c',
    'Impervious Surface (%)': 'impervious_pct',
    'PDO Winter Index':       'pdo_winter_mean',
    'Days Above 18°C':        'days_above_18c',
}

print('Mann-Kendall Trend Test Results (Issaquah Creek, 1985–2025)')
print('=' * 72)
print(f'{"Variable":<30} {"Trend":<12} {"p-value":<10} {"Sen Slope":<12} {"Significant"}')
print('-' * 72)

mk_results = []
for name, col in mk_targets.items():
    if col not in df.columns:
        continue
    series = df[col].dropna()
    if len(series) < 10:
        continue
    result = mk.original_test(series)
    sig = '*** p<0.001' if result.p < 0.001 else ('** p<0.01' if result.p < 0.01
           else ('* p<0.05' if result.p < 0.05 else 'ns'))
    print(f'{name:<30} {result.trend:<12} {result.p:<10.4f} {result.slope:<12.3f} {sig}')
    mk_results.append({
        'variable': name, 'trend': result.trend,
        'p_value': result.p, 'sen_slope': result.slope, 'significance': sig
    })

mk_df = pd.DataFrame(mk_results)
print('\n✓ Mann-Kendall tests complete.')
print('  Negative slope = declining over time; Positive slope = increasing.')

## 6. Before / After Urban Growth Analysis

In [ ]:
# Compare returns across three urbanization eras
era_map = {
    'Pre-2000\n(Slow Growth)':        (1985, 1999),
    '2000–2012\n(Sammamish Boom)':    (2000, 2012),
    '2013–2025\n(Continued Expansion)':(2013, 2025),
}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (species, col, color) in zip(axes, [
    ('Chinook', 'chinook_total', BLUE_DARK),
    ('Coho',    'coho_total',    SALMON),
]):
    if col not in df.columns:
        continue
    era_means = []
    era_labels = []
    for era_name, (start, end) in era_map.items():
        mean_val = df.loc[df['water_year'].between(start, end), col].mean()
        era_means.append(mean_val)
        era_labels.append(era_name)

    bars = ax.bar(era_labels, era_means, color=[color]*3,
                  alpha=[0.5, 0.7, 0.95], edgecolor='white', linewidth=1.5)

    for bar, val in zip(bars, era_means):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                f'{val:,.0f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

    ax.set_title(f'{species} — Average Annual Returns by Era', fontsize=11, fontweight='bold')
    ax.set_ylabel('Mean Annual Returns (fish)', fontsize=10)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

plt.suptitle('Issaquah Creek: Salmon Returns Before & After Sammamish Urbanization',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / '04_era_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → 04_era_comparison.png')

## 7. EDA Summary

*(Fill in findings after running the notebook on real data)*

| Finding | Value | Direction |
|---------|-------|-----------|
| Chinook 40-yr trend (Mann-Kendall) | TBD | TBD |
| Coho 40-yr trend (Mann-Kendall) | TBD | TBD |
| Snowpack trend (Apr 1 SWE) | TBD | TBD |
| Water temperature trend | TBD | TBD |
| Impervious surface growth | TBD | TBD |
| Strongest Chinook correlation | TBD | TBD |
| Strongest Coho correlation | TBD | TBD |
| Mean returns: pre-2000 vs post-2013 | TBD | TBD |

**➡ Proceed to `02_stats.ipynb` for statistical impact analysis.**